# 03 - Geological Diagenesis & Petrophysical Characterization
### poropack: 3D Porous Media & Digital Rock Physics Generator

This notebook demonstrates:
1. **Diagenetic History**: Simulating mechanical vertical compaction, syntaxial mineral cementation (EDT dilation), and secondary dissolution / leaching.
2. **Porosity Partitioning**: Analyzing total, percolating (effective), and isolated dead-end porosity.
3. **Pore Geometry Metrics**: Specific surface area $S_v$, two-point correlation function $S_2(r)$, and chord length distributions.
4. **Fluid Transport**: Kozeny-Carman permeability and Pore Network Modeling (PNM) via PoreSpy SNOW2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import poropack as pp

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11


## 1. Simulating Post-Depositional Diagenesis

Natural reservoir sandstones experience burial history:
- **Compaction**: Uniaxial vertical overburden stress squashes grain centroids and compresses the vertical axis ($z' = (1 - 
arepsilon_{zz}) z$).
- **Cementation**: Syntaxial quartz overgrowths and calcite cements precipitate outward from grain surfaces, preferentially clogging narrow pore throats.
- **Dissolution**: Acidic formation fluids leach unstable grains (e.g. feldspar leaching) creating secondary vugs and intra-granular pore networks.


In [ ]:
# Generate initial depositional pack
rsa = pp.RSAGenerator(
    box_size=(320.0, 320.0, 320.0),
    psd=pp.LogNormalPSD(d50=35.0, sigma_phi=0.25),
    random_state=42,
)
pack_init = rsa.generate(target_porosity=0.58)
grid_depositional = pack_init.rasterize(voxel_size=3.2)

# Stage 1: 15% Uniaxial Vertical Mechanical Compaction
grid_compacted = pp.apply_compaction(grid_depositional, vertical_strain=0.15)

# Stage 2: 10% Syntaxial Quartz Overgrowth Cementation (EDT dilation)
grid_cemented = pp.apply_cementation(grid_compacted, cement_fraction=0.10)

# Stage 3: 5% Secondary Feldspar Core Leaching / Dissolution
grid_dissolved = pp.apply_dissolution(grid_cemented, porosity_increase=0.05, mode='grain_cores')

print('Diagenetic Porosity Evolution Tracking:')
print(f'  1. Depositional Porosity: {grid_depositional.porosity():.4f}')
print(f'  2. Compacted Porosity:    {grid_compacted.porosity():.4f}')
print(f'  3. Cemented Porosity:     {grid_cemented.porosity():.4f}')
print(f'  4. Dissolved Porosity:    {grid_dissolved.porosity():.4f}')

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(grid_depositional.matrix[50, :, :], cmap='bone')
axes[0].set_title(f'1. Depositional\n$\\phi = {grid_depositional.porosity():.2f}$')
axes[1].imshow(grid_compacted.matrix[int(50*0.85), :, :], cmap='bone')
axes[1].set_title(f'2. Compacted\n$\\phi = {grid_compacted.porosity():.2f}$')
axes[2].imshow(grid_cemented.matrix[int(50*0.85), :, :], cmap='bone')
axes[2].set_title(f'3. Cemented\n$\\phi = {grid_cemented.porosity():.2f}$')
axes[3].imshow(grid_dissolved.matrix[int(50*0.85), :, :], cmap='bone')
axes[3].set_title(f'4. Dissolved\n$\\phi = {grid_dissolved.porosity():.2f}$')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()


## 2. Digital Rock Petrophysics Characterization

Evaluate key petrophysical properties:
- **Porosity Partitioning**: 3D connected component labeling checks whether pore clusters span boundary faces along X, Y, and Z.
- **Specific Surface Area ($S_v$)**: Surface area per unit volume computed via the Crofton intersection formula.
- **Two-Point Correlation Function $S_2(r)$**: Probability that two points separated by distance $r$ both fall within the void phase.
- **Chord Length Distributions**: 1D pore segment lengths.


In [ ]:
# Porosity partitioning
poro_res = pp.analyze_porosity(grid_cemented)
print(poro_res.summary())

# Specific surface area
sv = pp.specific_surface_area(grid_cemented)
print(f'Specific Surface Area (Sv): {sv:.4f} um^-1')

# Two-point correlation function S2(r)
radii_s2, s2_vals = pp.two_point_correlation(grid_cemented, max_radius=30)

# Chord length distribution
chords_pore = pp.chord_length_distribution(grid_cemented, phase='pore', axis=0)

# Kozeny-Carman Permeability
k_m2 = pp.kozeny_carman(grid_cemented)
k_mD = k_m2 / 9.869233e-16
print(f'Kozeny-Carman Permeability: {k_mD:.2f} mD')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(radii_s2 * grid_cemented.voxel_size, s2_vals, marker='o', color='crimson')
axes[0].set_title(r'Two-Point Autocorrelation Function $S_2(r)$')
axes[0].set_xlabel('Lag Distance r (um)')
axes[0].set_ylabel(r'$S_2(r)$')
axes[0].grid(True, alpha=0.3)

axes[1].hist(chords_pore * grid_cemented.voxel_size, bins=30, density=True, color='teal', alpha=0.75)
axes[1].set_title('Pore Chord Length Distribution')
axes[1].set_xlabel('Chord Length (um)')
axes[1].set_ylabel('Probability Density')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Pore Network Modeling (PNM) Extraction via PoreSpy SNOW2

Extract the topological network (pore bodies, throats, and connectivity) for pore-scale single and multiphase network flow simulations.


In [ ]:
try:
    net = pp.extract_pore_network(grid_cemented)
    print('SNOW2 Network Extraction Successful:')
    print(f'  Pore Bodies:           {len(net["pore.coords"])}')
    print(f'  Pore Throats:          {len(net["throat.conns"])}')
    print(f'  Mean Pore Diameter:    {2.0 * np.mean(net["pore.equivalent_diameter"]):.2f} um')
    print(f'  Mean Throat Diameter:  {2.0 * np.mean(net["throat.equivalent_diameter"]):.2f} um')
except Exception as e:
    print(f'Pore network extraction note: {e}')
